In [1]:
import pandas as pd 
import numpy as np 
import os

In [2]:
dir = "synthea-mimic/csv"
for file in os.listdir(dir):
    print(dir, file)
    data = pd.read_csv(os.path.join(dir, file), on_bad_lines='skip')
    print(file)
    print(data.columns)
    print()

synthea-mimic/csv allergies.csv
allergies.csv
Index(['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE', 'SYSTEM',
       'DESCRIPTION', 'TYPE', 'CATEGORY', 'REACTION1', 'DESCRIPTION1',
       'SEVERITY1', 'REACTION2', 'DESCRIPTION2', 'SEVERITY2'],
      dtype='object')

synthea-mimic/csv careplans.csv
careplans.csv
Index(['Id', 'START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION',
       'REASONCODE', 'REASONDESCRIPTION'],
      dtype='object')

synthea-mimic/csv claims.csv
claims.csv
Index(['Id', 'PATIENTID', 'PROVIDERID', 'PRIMARYPATIENTINSURANCEID',
       'SECONDARYPATIENTINSURANCEID', 'DEPARTMENTID', 'PATIENTDEPARTMENTID',
       'DIAGNOSIS1', 'DIAGNOSIS2', 'DIAGNOSIS3', 'DIAGNOSIS4', 'DIAGNOSIS5',
       'DIAGNOSIS6', 'DIAGNOSIS7', 'DIAGNOSIS8', 'REFERRINGPROVIDERID',
       'APPOINTMENTID', 'CURRENTILLNESSDATE', 'SERVICEDATE',
       'SUPERVISINGPROVIDERID', 'STATUS1', 'STATUS2', 'STATUSP',
       'OUTSTANDING1', 'OUTSTANDING2', 'OUTSTANDINGP', 'LASTBILLEDDATE1',
       

In [3]:
encounters = pd.read_csv(os.path.join(dir, "encounters.csv"), on_bad_lines='skip', usecols=["PATIENT", "START", "ENCOUNTERCLASS"])
encounters.describe()

,START,PATIENT,ENCOUNTERCLASS
count,126049,126049,126049
unique,123972,2275,11
top,1949-04-19T02:20:49Z,52a6c790-0b1d-299e-dbd0-60b9ebc89b49,ambulatory
freq,10,772,70213


In [4]:
first_encounter = encounters.groupby('PATIENT')['START'].min().reset_index()
first_encounter['START'] = pd.to_datetime(first_encounter['START'])
# picking median date of first encounter to split
cutoff_date = pd.to_datetime(first_encounter['START'].median())
print(cutoff_date, len(first_encounter))
#encounters['FIRST'] = pd.to_datetime(encounters['FIRST'])
first_encounter['WINDOW'] = first_encounter['START'].apply(
    lambda x: 'historical' if x < cutoff_date else 'current'
)

historical_patients = first_encounter[first_encounter['WINDOW'] == 'historical']['PATIENT']
current_patients    = first_encounter[first_encounter['WINDOW'] == 'current']['PATIENT']

1994-10-14 10:59:54+00:00 2275


In [5]:
historical_patients.describe()  

count                                     1137
unique                                    1137
top       ffc5d3f8-ce27-e72f-ec38-a0c250cb6833
freq                                         1
Name: PATIENT, dtype: object

In [6]:
current_patients.describe()

count                                     1138
unique                                    1138
top       ffc8bfa4-30fc-ec56-53ac-9f96a8ee482c
freq                                         1
Name: PATIENT, dtype: object

In [7]:
emergency_encounters_per_patient = encounters[encounters['ENCOUNTERCLASS'] == 'emergency'].groupby('PATIENT').size()
emergency_encounters_per_patient = emergency_encounters_per_patient.reset_index(name='EMERGENCY_ENCOUNTERS')

historical_data = pd.merge(historical_patients, emergency_encounters_per_patient, on='PATIENT', how='left').fillna(0)
current_data = pd.merge(current_patients, emergency_encounters_per_patient, on='PATIENT', how='left').fillna(0)

historical_data, current_data

(                                   PATIENT  EMERGENCY_ENCOUNTERS
 0     00a7caf4-3c83-32ac-2d3b-5458b2b3805f                   6.0
 1     00b74cc0-9354-07f8-fc2c-f7e64b4f9227                   4.0
 2     00c3d983-9b2a-f9e2-c622-b3eb5c960dbb                   4.0
 3     010e3238-3643-f735-4752-1dbd1b6e5cbc                   4.0
 4     011a092f-1d3a-cccf-31bb-1276d201cbf4                   0.0
 ...                                    ...                   ...
 1132  fecad4df-cb10-0c35-3afa-566c4ca1c0f3                   4.0
 1133  feef4595-6f66-f73b-0c16-6101b6284e36                   7.0
 1134  ff80137d-8184-4ee6-a6d6-c5f567d61c44                   0.0
 1135  ff91eb41-7717-d1f8-7119-b87035f8f0b9                   1.0
 1136  ffc5d3f8-ce27-e72f-ec38-a0c250cb6833                   1.0
 
 [1137 rows x 2 columns],
                                    PATIENT  EMERGENCY_ENCOUNTERS
 0     00043074-bbc9-7d65-195e-e615d7b61dcb                   2.0
 1     0033b4c0-255c-af66-8fc9-ebe17f725461     

In [8]:
conditions = pd.read_csv(os.path.join(dir, "conditions.csv"), on_bad_lines='skip', usecols=["PATIENT"])
conditions_per_patient = conditions.groupby('PATIENT').size().reset_index(name='CONDITIONS')
historical_data = pd.merge(historical_data, conditions_per_patient, on='PATIENT', how='left').fillna(0)
current_data = pd.merge(current_data, conditions_per_patient, on='PATIENT', how='left').fillna(0)
#historical_data.describe(), current_data.describe()

In [9]:
allergies = pd.read_csv(os.path.join(dir, "allergies.csv"), on_bad_lines='skip', usecols=["PATIENT"])
allergies_per_patient = allergies.groupby('PATIENT').size().reset_index(name='ALLERGIES')
historical_data = pd.merge(historical_data, allergies_per_patient, on='PATIENT', how='left').fillna(0)
current_data = pd.merge(current_data, allergies_per_patient, on='PATIENT', how='left').fillna(0)
#historical_data.describe(), current_data.describe()

In [10]:
medications = pd.read_csv(os.path.join(dir, "medications.csv"), on_bad_lines='skip', usecols=["PATIENT"])
medications_per_patient = medications.groupby('PATIENT').size().reset_index(name='MEDICATIONS')
historical_data = pd.merge(historical_data, medications_per_patient, on='PATIENT', how='left').fillna(0)
current_data = pd.merge(current_data, medications_per_patient, on='PATIENT', how='left').fillna(0)
#historical_data.describe(), current_data.describe()

In [11]:
procedures = pd.read_csv(os.path.join(dir, "procedures.csv"), on_bad_lines='skip', usecols=["PATIENT", 'BASE_COST'])
procedures['BASE_COST'] = pd.to_numeric(procedures['BASE_COST'], errors='coerce').fillna(0) 
procedures_cost_per_patient = procedures.groupby('PATIENT')['BASE_COST'].sum().reset_index(name='PROCEDURE_COST')
historical_data = pd.merge(historical_data, procedures_cost_per_patient, on='PATIENT', how='left').fillna(0)
current_data = pd.merge(current_data, procedures_cost_per_patient, on='PATIENT', how='left').fillna(0)
#historical_data.describe(), current_data.describe()

C:\Users\adity\AppData\Local\Temp\ipykernel_15280\984110011.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  procedures = pd.read_csv(os.path.join(dir, "procedures.csv"), on_bad_lines='skip', usecols=["PATIENT", 'BASE_COST'])


In [12]:
patients = pd.read_csv(os.path.join(dir, "patients.csv"), on_bad_lines='skip', 
                       usecols=["Id", "BIRTHDATE", "DEATHDATE", "INCOME", "HEALTHCARE_COVERAGE"])
patients = patients.rename(columns={'Id': 'PATIENT'})
patients['BIRTHDATE'] = pd.to_datetime(patients['BIRTHDATE'], errors='coerce')
patients['DEATHDATE'] = pd.to_datetime(patients['DEATHDATE'], errors='coerce')

#Remove .describe() here — it was converting the df to summary stats too early
historical_data = pd.merge(historical_data, patients, on='PATIENT', how='left')
current_data    = pd.merge(current_data,    patients, on='PATIENT', how='left')

today = pd.Timestamp('today')

#Use np.where for row-wise condition, and .dt.days AFTER subtraction
historical_data['AGE'] = np.where(
    historical_data['DEATHDATE'].isna(),
    (today - historical_data['BIRTHDATE']).dt.days,
    (historical_data['DEATHDATE'] - historical_data['BIRTHDATE']).dt.days
)//365

current_data['AGE'] = np.where(
    current_data['DEATHDATE'].isna(),
    (today - current_data['BIRTHDATE']).dt.days,
    (current_data['DEATHDATE'] - current_data['BIRTHDATE']).dt.days
)//365

historical_data = historical_data.drop(columns=['BIRTHDATE', 'DEATHDATE'])
current_data    = current_data.drop(columns=['BIRTHDATE', 'DEATHDATE'])

#.describe() only at the end for inspection
historical_data.describe(), current_data.describe()

(       EMERGENCY_ENCOUNTERS   CONDITIONS    ALLERGIES  MEDICATIONS  \
 count           1137.000000  1137.000000  1137.000000  1137.000000   
 mean               2.468777     1.861038     0.870712    78.659631   
 std                5.227664     9.457375     2.352328   174.415350   
 min                0.000000     0.000000     0.000000     0.000000   
 25%                0.000000     0.000000     0.000000     9.000000   
 50%                1.000000     0.000000     0.000000    24.000000   
 75%                3.000000     0.000000     0.000000    53.000000   
 max               84.000000   163.000000    13.000000  1672.000000   
 
        PROCEDURE_COST          AGE  
 count    1.137000e+03  1137.000000  
 mean     1.790272e+05    61.030783  
 std      3.188862e+05    15.861913  
 min      0.000000e+00     1.000000  
 25%      6.010960e+04    52.000000  
 50%      9.202394e+04    60.000000  
 75%      1.691467e+05    70.000000  
 max      3.342529e+06   111.000000  ,
        EMERGENC

In [13]:
historical_data['HIGH_EMERGENCY_RISK'] = (historical_data['EMERGENCY_ENCOUNTERS'] > 3).astype(int)
current_data['HIGH_EMERGENCY_RISK'] = (current_data['EMERGENCY_ENCOUNTERS'] > 3).astype(int)

In [14]:
historical_data.to_csv("data/historical_data.csv", index=False)
current_data.to_csv("data/current_data.csv", index=False)